In [259]:
from dash import Dash
from dash import dcc
from dash import html
from dash import State
from dash.dependencies import Input, Output
from pandas_datareader import data
import pandas as pd
import plotly.graph_objects as go




In [260]:

#import data here

df_fifa = pd.read_csv('/Users/husna/Downloads/BAS476/Fifa_world_cup_matches.csv')


print(df_fifa.head())

           team1         team2 possession team1 possession team2  \
0          QATAR       ECUADOR              42%              50%   
1        ENGLAND          IRAN              72%              19%   
2        SENEGAL   NETHERLANDS              44%              45%   
3  UNITED STATES         WALES              51%              39%   
4      ARGENTINA  SAUDI ARABIA              64%              24%   

  possession in contest  number of goals team1  number of goals team2  \
0                    8%                      0                      2   
1                    9%                      6                      2   
2                   11%                      0                      2   
3                   10%                      1                      1   
4                   12%                      1                      2   

          date     hour category  ...  penalties scored team1  \
0  20 NOV 2022  17 : 00  Group A  ...                       0   
1  21 NOV 2022  14 : 0

In [261]:
# created a dataframe here that shows the total # of goals per team, we were investigating the data and wanted to see if this
# could potentially be crucial information


# Step 1: Rename columns to represent each team's goals, then combine
goals_team1 = df_fifa[['team1', 'number of goals team1']].rename(columns={'team1': 'team', 'number of goals team1': 'goals'})
goals_team2 = df_fifa[['team2', 'number of goals team2']].rename(columns={'team2': 'team', 'number of goals team2': 'goals'})

# Step 2: Concatenate both dataframes to get a single 'team' and 'goals' column
total_goals_df = pd.concat([goals_team1, goals_team2])

# Step 3: Group by team and sum the goals for each team
team_goals = total_goals_df.groupby('team')['goals'].sum().reset_index()

# Display the aggregated goals by team
team_goals.columns = ['team', 'total_goals']
team_goals



,team,total_goals
0,ARGENTINA,15
1,AUSTRALIA,4
2,BELGIUM,1
3,BRAZIL,8
4,CAMEROON,4
5,CANADA,2
6,COSTA RICA,3
7,CROATIA,8
8,DENMARK,1
9,ECUADOR,4


In [262]:

# Here we wanted to create a column with the percentage of succeeded attempts based on other columns, but we did not end up using this in our dash

# Replace zeros with NaN to avoid errors
df_fifa['total attempts team1'] = df_fifa['total attempts team1'].replace(0, float('nan'))
df_fifa['total attempts team2'] = df_fifa['total attempts team2'].replace(0, float('nan'))

# Calculate the Succeeded Attempts as percentages
df_fifa['Succeeded Attempts team1 (%)'] = ((df_fifa['number of goals team1'] / df_fifa['total attempts team1']) * 100).fillna(0).round().astype(int).astype(str) + '%'
df_fifa['Succeeded Attempts team2 (%)'] = ((df_fifa['number of goals team2'] / df_fifa['total attempts team2']) * 100).fillna(0).round().astype(int).astype(str) + '%'




In [263]:


# Combine the total goals from team1 and team2 into a single 'total_goals' column
df_fifa['total_goals'] = df_fifa['number of goals team1'] + df_fifa['number of goals team2']




# Create column that has list of all unique teams by concatenating team1 and team2 columns
all_teams = pd.concat([df_fifa['team1'], df_fifa['team2']], ignore_index=True)

# Remove duplicates
all_teams_unique = all_teams.drop_duplicates().reset_index(drop=True)

# store this list as a new DataFrame column
df_fifa['all_teams'] = pd.Series(all_teams_unique)

print(df_fifa['all_teams'].head()) 



# more data clean up

#create column with just year
df_fifa['year'] = pd.to_datetime(df_fifa['date']).dt.year

# Assuming df_fifa is your dataframe
df_fifa['date'] = pd.to_datetime(df_fifa['date'])  # Ensure 'date' column is datetime type

# Create a new column for total goals (sum of goals for both teams)
df_fifa['total_goals'] = df_fifa['number of goals team1'] + df_fifa['number of goals team2']

# Create a new column for match label (team1 vs team2)
df_fifa['match'] = df_fifa['team1'] + " vs " + df_fifa['team2']

# Ensure that matches are in the correct order (team1 vs team2), sorting is important
df_fifa_sorted = df_fifa.sort_values(by='date')



0            QATAR
1          ENGLAND
2          SENEGAL
3    UNITED STATES
4        ARGENTINA
Name: all_teams, dtype: object


/var/folders/st/_bddt9j94fvg2t3j5dq5mqzw0000gn/T/ipykernel_96434/2063677806.py:23: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/var/folders/st/_bddt9j94fvg2t3j5dq5mqzw0000gn/T/ipykernel_96434/2063677806.py:26: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [264]:
# Layout Styles

df_fifa['total_goals'] = df_fifa['number of goals team1'] + df_fifa['number of goals team2']

style_for_label = dict(
    color="maroon", 
    fontWeight="bold", 
    borderRadius='5px', 
    fontSize=25, 
    padding='5px'
)
style_for_input = dict(
    width='200px', 
    height='30px', 
    border='2px solid blue', 
    borderRadius='5px', 
    padding='5px', 
    fontSize=20, 
    margin='10px'
)
style_for_button = dict(
    width='400px', 
    height='60px', 
    border='2px solid blue', 
    borderRadius='5px', 
    background="skyblue", 
    color="blue", 
    fontWeight="bold", 
    fontSize=20, 
    margin='10px'
)

# Initial Graph

df_short = df_fifa[df_fifa['year'] == 2022].sort_values(by='total_goals', ascending=False)[:10]  # Adjust for your data

fig = go.Figure([
    go.Bar(
        x=df_short['team1'] + " vs " + df_short['team2'], 
        y=df_short['total_goals'], 
        marker=dict(
            color=df_short['total_goals'], 
            colorscale='Viridis'
        )
    )
])
fig.update_layout(
    title="FIFA Matches in 2022",
    title_x=0.5,
    width=1000,
    height=700,
    font=dict(color="purple", size=25),
    xaxis=dict(
        title="Match (Team1 vs Team2)",
        title_font=dict(size=20, color="black"),
        tickangle=-45
    ),
    yaxis=dict(
        title="Total Goals",
        title_font=dict(size=20, color="black")
    ),
    margin=dict(l=50, r=50, t=100, b=100),
    plot_bgcolor="rgba(240, 240, 240, 0.8)"
)


In [265]:



# Mapping country names to ISO country codes for choropleth map
country_code_mapping = {
    'QATAR': 'QAT', 'ENGLAND': 'GBR', 'SENEGAL': 'SEN', 'UNITED STATES': 'USA', 'ARGENTINA': 'ARG', 'AUSTRALIA': 'AUS', 'BELGIUM': 'BEL', 'BRAZIL': 'BRA', 'CAMEROON': 'CMR', 'CANADA': 'CAN',
    'COSTA RICA': 'CRI', 'CROATIA': 'HRV', 'DENMARK': 'DNK', 'ECUADOR': 'ECU', 'FRANCE': 'FRA', 'GERMANY':'DEU', 'GHANA': 'GHA', 'IRAN': 'IRN', 'JAPAN': 'JPN', 'KOREA REPUBLIC': 'KOR', 
    'MEXICO': 'MEX', 'MOROCCO': 'MAR', 'NETHERLANDS': 'NLD', 'POLAND': 'POL', 'PORTUGAL': 'PRT', 'SAUDI ARABIA': 'SAU', 'SERBIA': 'SRB', 'SPAIN': 'ESP', 'SWITZERLAND': 'CHE', 'TUNISIA': 'TUN', 
     'URUGUAY': 'URY', 'WALES': 'WLS'
   
}

# Merge the team data with the country codes
team_goals['iso_alpha'] = team_goals['team'].map(country_code_mapping)

# Create the choropleth map for total goals by country
choromap = go.Figure(go.Choropleth(
    z=team_goals['total_goals'],  # extract total goals data
    locations=team_goals['iso_alpha'],  # ISO country codes
    colorbar_title="Total Goals",  # Title
    hovertext=team_goals['team'],  # what comes up when hover
))

# Layout for choropleth map
layout_world = dict(
    geo=dict(
        scope='world',  # whole world as scope
        projection=dict(type='natural earth'),  # natural earth setting
        showcoastlines=True,
        coastlinecolor="Black",  
        projection_scale=4,  # Scale for the projection
        showland=True,
        landcolor='rgb(255, 255, 255)'  # White land color for better contrast
    ),
    title='World Goals Map <br> (Hover for total goals)',
    autosize=False,
    width=800,
    margin=dict(l=20, r=20, t=50, b=50),
)


choromap.update_layout(layout_world)  


choromap.show()


In [266]:
# This is the one that works

# Placeholder choropleth map (replace `choromap` with your map figure object)
choromap = go.Figure(go.Choropleth(
    z=team_goals['total_goals'],  # The data for the total goals
    locations=team_goals['iso_alpha'],  # The ISO country codes
    colorbar_title="Total Goals",  # Color bar title
    hovertext=team_goals['team'],  # Show the country name when hovered
))


# Initialize Dash App
app = Dash("FIFA Analysis")

# Layout Styles


style_for_button = dict(
    width='200px',  # Increased width to make the button longer
    height='50px',
    border='2px solid purple',
    borderRadius='5px',
    background="#E6E6FA",
    color="purple",
    fontWeight="bold",
    fontSize=20,
    margin='20px'
)

style_for_first_button = dict(
    width='300px',  # Increased width to make the button longer
    height='200px',
    border='4px solid purple',
    borderRadius='5px',
    background="#E6E6FA",
    color="purple",
    fontWeight="bold",
    fontSize=20,
    margin='20px'
)



# Layout Components
header = html.H1("FIFA Data Dashboard", style={'textAlign': 'center', 'color': 'purple'})
horizontal_line = html.Hr()
sub_header = html.H2(
    "Outcome of Matches in 2022: Who Won?",
    style={
        'textAlign': 'left', 
        'color': 'white',  # Change text color for contrast
        'backgroundColor': 'purple',  # Add background color
        'fontSize': '25px',  # Make the text bigger
        'fontWeight': 'bold',  # Make the text bold
        'padding': '10px',  # Add padding around the text
        'borderRadius': '8px',  # Optional: round the corners of the background
        'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.2)',  # Optional: add a shadow for a "lifted" effect
        'marginBottom': '20px'  # Add some space below the header
    }
)
increase_button = html.Button(
    id="change_teams_button",
    n_clicks=0,
    children="Change Teams",
    style=style_for_first_button
)
graph_container = dcc.Graph(id='team_goals_graph')

choropleth_header = html.H2("FIFA Goals Choropleth Map", style={
        'textAlign': 'left', 
        'color': 'white',  # Change text color for contrast
        'backgroundColor': 'purple',  # Add background color
        'fontSize': '25px',  # Make the text bigger
        'fontWeight': 'bold',  # Make the text bold
        'padding': '10px',  # Add padding around the text
        'borderRadius': '8px',  # Optional: round the corners of the background
        'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.2)',  # Optional: add a shadow for a "lifted" effect
        'marginBottom': '20px'  # Add some space below the header
    })
map_container = dcc.Graph(figure=choromap)

analysis_header = html.H2("Analyze Matches and Corresponding Stats", style={
        'textAlign': 'left', 
        'color': 'white',  # Change text color for contrast
        'backgroundColor': 'purple',  # Add background color
        'fontSize': '25px',  # Make the text bigger
        'fontWeight': 'bold',  # Make the text bold
        'padding': '10px',  # Add padding around the text
        'borderRadius': '8px',  # Optional: round the corners of the background
        'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.2)',  # Optional: add a shadow for a "lifted" effect
        'marginBottom': '20px'  # Add some space below the header
    })
analysis_input = html.Div(children=[
    html.Label("Enter Team 1:", style={'fontWeight': 'bold', 'fontSize': '20px'}),
    dcc.Input(id='team1_input', type='text', placeholder='Team 1', style={'margin': '10px'}),
    html.Label("Enter Team 2:", style={'fontWeight': 'bold', 'fontSize': '20px'}),
    dcc.Input(id='team2_input', type='text', placeholder='Team 2', style={'margin': '10px'}),
    html.Button("Analyze Match", id='analyze_button', n_clicks=0, style=style_for_button),
], style={'margin': '20px'})

analysis_output = html.Div(id='output_container', style={'margin': '20px', 'fontSize': '18px'})
analysis_graph = dcc.Graph(id='possession_graph', style={'height': '600px', 'width': '80%', 'margin': 'auto'})




stat_dropdown = html.Div(children= [
    html.Label("Select a Statistic to Analyze Below", style={'fontWeight': 'bold', 'fontSize': '20px'}),
    dcc.Dropdown(   id='stat_dropdown',
                    options=[   {'label': 'Goals', 'value': 'Goals'},
                                {'label': 'Total Attempt at Goal', 'value': 'Total Attempts at Goal'},
                                {'label': 'Total Offers to Receive', 'value': 'Total Offers to Receive'},
                                {'label': 'Completed Line Breaks', 'value': 'Completed Line Breaks'},
                                {'label': 'Yellow Cards', 'value': 'Yellow Cards'},  
                                {'label': 'Red Cards', 'value': 'Red Cards'},
                                {'label': 'Fouls Taken', 'value': 'Fouls Taken'},
                                {'label': 'Passes Completed', 'value': 'Passes Completed'},
                                {'label': 'Free Kicks', 'value': 'Free Kicks'},
                                {'label': 'Penalties', 'value': 'Penalties'},
                                {'label': 'Goal Preventions', 'value': 'Goal Preventions'}, ],
                    value='goals',  # Default selected value
                    placeholder="Select a Stat" )
],  style={'margin': '20px'})


team_analysis_header = html.H2("Let's Learn More About a Specific Team", style={
        'textAlign': 'left', 
        'color': 'white',  # Change text color for contrast
        'backgroundColor': 'purple',  # Add background color
        'fontSize': '25px',  # Make the text bigger
        'fontWeight': 'bold',  # Make the text bold
        'padding': '10px',  # Add padding around the text
        'borderRadius': '8px',  # Optional: round the corners of the background
        'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.2)',  # Optional: add a shadow for a "lifted" effect
        'marginBottom': '20px'  # Add some space below the header
    })
team_input = html.Div(children=[
    html.Label("Enter Team Name:", style={'fontWeight': 'bold', 'fontSize': '20px'}),
    dcc.Input(id='team_input', type='text', placeholder="Enter team name", style={'margin': '10px'}),
    html.Button('Analyze', id='analyze_team_button', n_clicks=0, style = style_for_button)], style={'margin': '20px'})

area = html.Div(id='match_display')
graph_for_team = dcc.Graph(id='match_graph')

app.layout = html.Div(
    children=[
        header,
        horizontal_line,
        html.Div(
            children=[
                # First column: sub_header and increase_button
                html.Div(sub_header, style={'gridColumn': '1 / 2 ', 'gridRow': '1', 'textAlign': 'left','marginBottom': '5px'}),
                html.Div(increase_button, style={'gridColumn': '1 / 2', 'gridRow': '3', 'textAlign': 'left', 'marginTop': '5px', 'marginBottom': '20px'}),

                # Result text (added below the increase_button)
                html.Div(id='match_result', style={'gridColumn': '1 / 2', 'gridRow': '2', 'textAlign': 'left', 'marginTop': '5px', 'marginBottom': '2px'}),

                # Second column: graph_container
                html.Div(graph_container, style={'gridColumn': '2 / 3', 'gridRow': '1 / 3', 'marginBottom': '50px'}),  # Span across rows
            ],
            style={
                'display': 'grid',
                'gridTemplateColumns': '1fr 2fr',  # First column smaller, second larger for the graph
                'gridTemplateRows': '100px 100px 320px',  # Three rows: one for sub_header, one for button, one for result
                'gap': '30px',                     # Spacing between rows and columns
                'alignItems': 'start',             # Align items to the top of each cell
                'marginBottom': '20px',            # Spacing below the grid
            }
        ),
        horizontal_line,
        analysis_header,
        analysis_input,
        stat_dropdown,
        analysis_output,
        analysis_graph,
        horizontal_line,
        team_analysis_header,
        team_input,
        area,
        graph_for_team,
        horizontal_line,
        choropleth_header,
        map_container,
        horizontal_line,
    ]
)

# Callback for Changing Team Goals Graph (with specific match goals)
@app.callback(
    [Output('team_goals_graph', 'figure'),
     Output('match_result', 'children')],  # Add Output for the result text
    [Input('change_teams_button', 'n_clicks')]
)
def update_graph(n_clicks):
    # Get pairs of teams that have actually played each other
    played_teams = set(
        tuple(sorted([row['team1'], row['team2']])) 
        for index, row in df_fifa.iterrows()
    )

    # Convert the set to a list of team pairs
    played_team_pairs = list(played_teams)

    # Ensure there are pairs to choose from
    if not played_team_pairs:
        return go.Figure(), None  # No result text if no matches

    # Select the next pair based on the number of clicks
    selected_pair = played_team_pairs[n_clicks % len(played_team_pairs)]
    team1, team2 = selected_pair

    # Initialize goals to 0
    team1_goals = 0
    team2_goals = 0

    # Iterate through matches between the two teams and get goals scored
    for _, row in df_fifa.iterrows():
        # Check if the pair of teams played each other
        if (row['team1'] == team1 and row['team2'] == team2) or (row['team1'] == team2 and row['team2'] == team1):
            # Add goals for team1 and team2 based on their respective positions
            if row['team1'] == team1:
                team1_goals += row['number of goals team1']
                team2_goals += row['number of goals team2']
            else:  # If team1 is actually team2 in this match
                team1_goals += row['number of goals team2']
                team2_goals += row['number of goals team1']

    # Create the bar graph
    fig = go.Figure([
        go.Bar(
            x=[team1, team2],
            y=[team1_goals, team2_goals],
            marker=dict(color=[team1_goals, team2_goals], colorscale='Viridis')
        )
    ])

    # Update graph layout
    fig.update_layout(
        title=f"{team1} vs {team2}",
        title_x=0.5,
        xaxis_title="Teams",
        yaxis_title="Number of Goals",
        font=dict(size=18),
        plot_bgcolor="rgba(240, 240, 240, 0.8)",
        width=800,
        height=600
    )
    
    # Determine who won
    if team1_goals > team2_goals:
        result = f"{team1} won with {team1_goals} goals!"
        result_style = {
            'color': 'white',  # White text for contrast
            'backgroundColor': '#228B22',  # Green background (team1 wins)
            'fontSize': '18px',
            'fontWeight': 'bold',
            'padding': '10px',
            'borderRadius': '5px',
            'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.1)'  # Optional: shadow effect
        }
    elif team2_goals > team1_goals:
        result = f"{team2} won with {team2_goals} goals!"
        result_style = {
            'color': 'white',  # White text for contrast
            'backgroundColor': '##228B22',  # Green background (team1 wins)
            'fontSize': '18px',
            'fontWeight': 'bold',
            'padding': '10px',
            'borderRadius': '5px',
            'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.1)'  # Optional: shadow effect
        }
    else:
        result = f"The match ended in a draw with both teams scoring {team1_goals} goals."
        result_style = {
            'color': 'black',  # Neutral text color
            'backgroundColor': '#6c757d',  # Gray background for draw
            'fontSize': '18px',
            'fontWeight': 'bold',
            'padding': '10px',
            'borderRadius': '5px',
            'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.1)'  # Optional: shadow effect
        }

    # Return the result text along with the graph figure
    result_div = html.Div(result, style=result_style)
    return fig, result_div







@app.callback(
    [Output(analysis_output, 'children'),
     Output(analysis_graph, 'figure')],
    [Input('analyze_button', 'n_clicks')],
    [State('team1_input', 'value'), State('team2_input', 'value'),
     State('stat_dropdown', 'value')]  # Add stat dropdown as state
)
def analyze_match(n_clicks, team1, team2, selected_stat):
    if not team1 or not team2:
        return "Please enter both teams to analyze.", go.Figure()

    # Filter the match data for the selected teams
    match = df_fifa[
        ((df_fifa['team1'] == (team1)) & (df_fifa['team2'] == (team2))) |
        ((df_fifa['team1'] == (team2)) & (df_fifa['team2'] == (team1)))
    ]

    if match.empty:
        return f"No match found between {team1} and {team2}.", go.Figure()

    row = match.iloc[0]  # Get the first match (assuming only one match for simplicity)

    # Dynamically select the stat based on the dropdown value
    if selected_stat == 'Goals':
        # Get the goals for each team in the match (from number of goals team1 and team2)
        team1_stat = row['number of goals team1'] if row['team1'] == team1 else row['number of goals team2']
        team2_stat = row['number of goals team2'] if row['team1'] == team1 else row['number of goals team1']
    elif selected_stat == 'Total Attempts at Goal':
        # Get the total attempts for each team in the match (from total attempts team1 and team2)
        team1_stat = row['total attempts team1'] if row['team1'] == team1 else row['total attempts team2']
        team2_stat = row['total attempts team2'] if row['team1'] == team1 else row['total attempts team1']
    elif selected_stat == 'Total Offers to Receive':
        # Get the shots on total offers to receive
        team1_stat = row['total offers to receive team1'] if row['team1'] == team1 else row['total offers to receive team2']
        team2_stat = row['total offers to receive team2'] if row['team1'] == team1 else row['total offers to receive team1']
    elif selected_stat == 'Completed Line Breaks':
        # Get the complete line breaks 
        team1_stat = row['completed line breaksteam1'] if row['team1'] == team1 else row['completed line breaks team2']
        team2_stat = row['completed line breaks team2'] if row['team1'] == team1 else row['completed line breaksteam1']
    elif selected_stat == 'Yellow Cards':
        # Get the possession for each team in the match (from possession team1 and possession team2)
        team1_stat = row['yellow cards team1'] if row['team1'] == team1 else row['yellow cards team2']
        team2_stat = row['yellow cards team2'] if row['team1'] == team1 else row['yellow cards team1']
    elif selected_stat == 'Red Cards':
        # Get the possession for each team in the match (from possession team1 and possession team2)
        team1_stat = row['red cards team1'] if row['team1'] == team1 else row['red cards team2']
        team2_stat = row['red cards team2'] if row['team1'] == team1 else row['red cards team1']
    elif selected_stat == 'Fouls Taken':
        # Get the possession for each team in the match (from possession team1 and possession team2)
        team1_stat = row['fouls against team1'] if row['team1'] == team1 else row['fouls against team2']
        team2_stat = row['fouls against team2'] if row['team1'] == team1 else row['fouls against team1']
    elif selected_stat == 'Passes Completed':
        # Get the possession for each team in the match (from possession team1 and possession team2)
        team1_stat = row['passes completed team1'] if row['team1'] == team1 else row['passes completed team2']
        team2_stat = row['passes completed team2'] if row['team1'] == team1 else row['passes completed team1']
    elif selected_stat == 'Free Kicks':
        # Get the possession for each team in the match (from possession team1 and possession team2)
        team1_stat = row['free kicks team1'] if row['team1'] == team1 else row['free kicks team2']
        team2_stat = row['free kicks team2'] if row['team1'] == team1 else row['free kicks team1']
    elif selected_stat == 'Penalties':
        # Get the possession for each team in the match (from possession team1 and possession team2)
        team1_stat = row['penalties scored team1'] if row['team1'] == team1 else row['penalties scored team2']
        team2_stat = row['penalties scored team2'] if row['team1'] == team1 else row['penalties scored team1']
    elif selected_stat == 'Goal Preventions':
        # Get the possession for each team in the match (from possession team1 and possession team2)
        team1_stat = row['goal preventions team1'] if row['team1'] == team1 else row['goal preventions team2']
        team2_stat = row['goal preventions team2'] if row['team1'] == team1 else row['goal preventions team1']

    # Convert stats to integers for bar plot
    try:
        team1_stat = int(team1_stat)  # Ensure the value is an integer
        team2_stat = int(team2_stat)  # Ensure the value is an integer
    except ValueError:
        return "Invalid data for the selected stat.", go.Figure()
    
    # Determine the winner or if it's a draw
    if team1_stat > team2_stat:
        winner = f"{team1} had better {selected_stat} with {team1_stat}."
        winner_style = {
            'color': 'purple', 
            'backgroundColor': '#E6E6FA',  
            'fontSize': '18px',
            'fontWeight': 'bold',
            'padding': '10px',
            'borderRadius': '5px',
            'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.1)'  # Optional: shadow effect
        }
    elif team1_stat < team2_stat:
        winner = f"{team2} had better {selected_stat} with {team2_stat}."
        winner_style = {
            'color': 'purple', 
            'backgroundColor': '#E6E6FA',  
            'fontSize': '18px',
            'fontWeight': 'bold',
            'padding': '10px',
            'borderRadius': '5px',
            'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.1)'  # Optional: shadow effect
        }
    else:
        winner = f"The match ended in a draw in terms of {selected_stat}."
        winner_style = {
            'color': 'black',  # Neutral text color
            'backgroundColor': '#6c757d',  # Gray background for draw
            'fontSize': '18px',
            'fontWeight': 'bold',
            'padding': '10px',
            'borderRadius': '5px',
            'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.1)'  # Optional: shadow effect
        }




    

    # Create the output text (e.g., winner and stat comparison)
    if team1_stat > team2_stat:
        winner = f"{team1} had better {selected_stat} with {team1_stat}."
    elif team1_stat < team2_stat:
        winner = f"{team2} had better {selected_stat} with {team2_stat}."
    else:
        winner = f"The match ended in a draw in terms of {selected_stat}."

    # Create a bar graph for stat comparison
    fig = go.Figure([
        go.Bar(
            x=[team1, team2],
            y=[team1_stat, team2_stat],  # Use the integer values for the bars
            text=[f"{team1_stat}", f"{team2_stat}"],  # Show values on the bars
            textposition='auto',
            marker=dict(color=['blue', 'green'])
        )
    ])

    fig.update_layout(
        title=f"{selected_stat} Comparison",
        xaxis=dict(
            title="Teams",
            tickmode='array',
            tickvals=[team1, team2],  # Labels for teams
            ticktext=[team1, team2]  # Add custom tick text
        ),
        yaxis=dict(
            title=f"{selected_stat}",
            showgrid=True  # Show gridlines to make comparison easier
        ),
        font=dict(size=18),  # Set font size for readability
        plot_bgcolor="rgba(240, 240, 240, 0.8)",  # Set a light background color
        width=800,  # Set graph width
        height=600  # Set graph height
    )

    
# Return the styled output along with the winner message
    return html.Div(winner, style=winner_style), fig


@app.callback(
    [Output('match_display', 'children'),
     Output('match_graph', 'figure')],
    [Input('analyze_team_button', 'n_clicks')],
    [State('team_input', 'value')]
)
def display_team_matches(n_clicks, team):
    if not team:
        return "Please enter a team name.", go.Figure()

    # Filter matches where the team is either team1 or team2
    matches = df_fifa[
        (df_fifa['team1'] == team) | (df_fifa['team2'] == team)
    ]

    if matches.empty:
        return f"No matches found for {team}.", go.Figure()

    # Function to determine the match result for the inputted team
    def get_match_result(team, match):
        if match['team1'] == team:
            if match['number of goals team1'] > match['number of goals team2']:
                return 'win'
            elif match['number of goals team1'] < match['number of goals team2']:
                return 'lose'
            else:
                return 'draw'
        elif match['team2'] == team:
            if match['number of goals team2'] > match['number of goals team1']:
                return 'win'
            elif match['number of goals team2'] < match['number of goals team1']:
                return 'lose'
            else:
                return 'draw'
        return 'draw'  # Default return in case the team isn't part of the match

    # Prepare the match information with dynamic styles based on the input team result
    match_info = []
    for _, row in matches.iterrows():
        result = get_match_result(team, row)  # Get the result for the input team
        
        # Format the match info and set style based on result
        if result == 'win':
            style = {'backgroundColor': '#228B22', 'color': 'white'}
        elif result == 'lose':
            style = {'backgroundColor': '#B22222', 'color': 'white'}
        else:
            style = {'backgroundColor': '#e0e0e0', 'color': 'black'}  # Draw match
        
        match_info.append(
            html.Li(
                f"{row['date'].strftime('%Y-%m-%d')}: {row['team1']} vs {row['team2']} - {result.capitalize()}",
                style={
                    'padding': '10px',
                    'marginBottom': '8px',
                    'backgroundColor': style['backgroundColor'],
                    'color': style['color'],
                    'border': '1px solid #ddd',
                    'borderRadius': '5px',
                    'boxShadow': '0px 4px 6px rgba(0, 0, 0, 0.1)',
                    'fontWeight': 'bold',
                    'textAlign': 'left',
                    'cursor': 'pointer',
                    'transition': 'background-color 0.3s ease',  # Smooth transition for background color
                }
            )
        )
    

    


    




    # Create a new column to indicate the result of the match (Winner, Loser, Draw)
    matches['result'] = matches.apply(lambda row: 'Winner' if (row['team1'] == team and row['number of goals team1'] > row['number of goals team2']) or (row['team2'] == team and row['number of goals team2'] > row['number of goals team1']) 
                                    else ('Loser' if (row['team1'] == team and row['number of goals team1'] < row['number of goals team2']) or (row['team2'] == team and row['number of goals team2'] < row['number of goals team1']) 
                                    else 'Draw'), axis=1)

    # Filter the matches where the team is involved (either as team1 or team2)
    team_matches = matches[(matches['team1'] == team) | (matches['team2'] == team)]

    # Create the plot for the selected team
    fig = go.Figure()

    # Add the match results to the plot
    fig.add_trace(go.Scatter(
        x=team_matches['date'],
        y=team_matches['result'],  # The y-values now show the match result
        mode='lines+markers',
        name=f'{team} Matches',
        text=team_matches['match'],
        hoverinfo='text+y',
        marker=dict(color='black', size=12),
        line=dict(color='purple', width=2)
    ))

    # Update the layout for the plot
    fig.update_layout(
        title= f'Matches Played by {str.title(team)} Over Time',
        xaxis_title='Date',
        yaxis_title='Match Result',
        xaxis=dict(showgrid=True, tickangle=45),
        yaxis=dict(showgrid=True, tickvals=['Winner', 'Loser', 'Draw'], ticktext=['Winner', 'Loser', 'Draw']),  # Display match results on the y-axis
        font=dict(size=14),
        plot_bgcolor="rgba(240, 240, 240, 0.8)",
        width=800,
        height=600
    )


    

    


    return html.Ul(match_info), fig




if __name__ == '__main__':
    app.run_server(debug=True, host="127.0.0.1", port=8050)

    app.run(jupyter_mode="tab")

Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>

/var/folders/st/_bddt9j94fvg2t3j5dq5mqzw0000gn/T/ipykernel_96434/1276264844.py:529: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

